# 03 - Agent Demo

This notebook demonstrates the LangGraph ReAct agent that uses amendment-aware LlamaIndex RAG
to answer questions about indexed contracts.

## Architecture
```
User Query -> LangGraph Agent -> search_contracts tool -> LlamaIndex Query Engine -> Pinecone
                    ^                                           |
                    |___________ synthesized answer ____________|
```

The query engine is **amendment-aware**: when terms conflict between a base contract
and its amendments, it prioritizes the latest version (highest version / latest effective_date).

**Prerequisites:** Run `01_generate_data.ipynb` and `02_ingestion.ipynb` first.

In [1]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
from dotenv import load_dotenv

# Load .env from project root (needed when running from notebooks/ directory)
load_dotenv(Path('../.env'))

from contract_lens.config import get_settings
from contract_lens.observability import init_observability

settings = get_settings()
init_observability(settings)

## Build the Agent

The agent consists of:
- **Azure OpenAI LLM** with tool-calling capability
- **`search_contracts`** tool wrapping the LlamaIndex query engine
- **LangGraph** state machine with ReAct routing (agent -> tools -> agent -> END)

In [2]:
from contract_lens.agent.graph import build_agent

agent, callback_handler = build_agent(settings)
config = {'callbacks': [callback_handler]} if callback_handler else {}

print('Agent ready.')
print(f'LangFuse tracing: {"enabled" if callback_handler else "disabled"}')

Agent ready.
LangFuse tracing: enabled


## Helper Function

A utility to invoke the agent and display the full message trace.

In [3]:
from langchain_core.messages import HumanMessage


def ask(question: str) -> str:
    """Send a question to the agent and print the conversation trace."""
    print(f'User: {question}')
    print('-' * 60)

    result = agent.invoke(
        {'messages': [HumanMessage(content=question)]},
        config=config,
    )

    for msg in result['messages']:
        role = msg.__class__.__name__.replace('Message', '')
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f'  [{role}] Tool call: {tc["name"]}({tc["args"]})')
        elif role == 'Tool':
            preview = msg.content[:200] + '...' if len(msg.content) > 200 else msg.content
            print(f'  [{role}] {preview}')
        elif role != 'Human':
            print(f'  [{role}] {msg.content}')

    print('-' * 60)
    answer = result['messages'][-1].content
    return answer

## Example Queries

### Query 1: Current terms (should return latest amendment, not base)

In [4]:
ask('What is the current hourly rate for a Senior Architect in the IT service agreement?');

User: What is the current hourly rate for a Senior Architect in the IT service agreement?
------------------------------------------------------------


  [AI] Tool call: search_contracts({'query': 'Senior Architect hourly rate', 'section_type': 'payment'})
  [Tool] The hourly rate for a “Senior Architect” is not stated in the provided payment terms. The contract refers to a pricing schedule in Annex A, but Annex A (and any role-based rate table) is not included ...
  [AI] The IT service agreement text available in the repository does **not state a specific hourly rate for a “Senior Architect.”** It says the rates are defined in a **pricing schedule / rate card in Annex A**, but that annex (with the role-based hourly rate table) isn’t included in the indexed content I can access.

If you can share Annex A (or tell me the contract ID / attach the document), I can extract the current Senior Architect hourly rate from it.
------------------------------------------------------------


### Query 2: Amendment history

In [5]:
ask('Show me the history of pricing changes in the IT service agreement ITSVC-001.');

User: Show me the history of pricing changes in the IT service agreement ITSVC-001.
------------------------------------------------------------


  [AI] Tool call: search_contracts({'query': 'pricing changes history amendment price fee rates remuneration', 'contract_id': 'ITSVC-001', 'section_type': 'payment'})
  [Tool] For contract **ITSVC-001**, there are **no amendments shown that change pricing, fees, rates, or remuneration amounts**.

What changed in the payment-related terms:
- **Invoice frequency:** changed fr...
  [AI] For **ITSVC-001**, the indexed contract history shows **no amendments that change pricing amounts** (fees/rates/remuneration). The **pricing basis remains per the “pricing schedule in Annex A”**, and no revised Annex A/pricing table is recorded as an amendment.

### Payment/pricing-related change history (what did change)
- **Effective 2026-01-01**: billing and payment terms were amended:
  - **Invoice frequency**: **Monthly → Bi-weekly**
  - **Payment due date**: **Net 30 (within 30 days of receipt) → Net 14 (within 14 days of receipt)**

### Explicitly unchanged (per available documents)
- **Late payment

### Query 3: SLA targets (should return amended values)

In [6]:
ask('What is the P1 incident response time in the Cloud SLA?');

User: What is the P1 incident response time in the Cloud SLA?
------------------------------------------------------------


  [AI] Tool call: search_contracts({'query': 'P1 incident response time', 'contract_id': 'SLA-001', 'section_type': 'sla'})
  [Tool] P1 incidents are classified as **Critical**. The **response time target for P1** is **defined in Annex A** (response and resolution targets per priority are set out there).
  [AI] Tool call: search_contracts({'query': 'Annex A response time P1 Critical response time target', 'contract_id': 'SLA-001', 'has_table': 'true'})
  [Tool] The Annex A **P1 – Critical** incident **response time target is 5 minutes**.
  [AI] In the Cloud SLA (Annex A), the **P1 (Critical) incident response time target is 5 minutes**.
------------------------------------------------------------


### Query 4: Polish language — lease amendment

In [7]:
ask('Jaki jest aktualny czynsz najmu biura i co zmienilo sie w aneksie?');

User: Jaki jest aktualny czynsz najmu biura i co zmienilo sie w aneksie?
------------------------------------------------------------


  [AI] Tool call: search_contracts({'query': 'czynsz najmu biura aneks zmiana czynszu', 'language': 'pl', 'section_type': 'payment'})
  [Tool] Aneks do umowy LEASE-001 zmienia punkt 3.1 w ten sposób, że **miesięczny czynsz najmu biura wynosi 28 500 PLN netto + VAT 23%** (zamiast 25 000 PLN netto + VAT 23% w umowie bazowej). Aneks obowiązuje ...
  [AI] Tool call: search_contracts({'query': 'LEASE-001 3.1 czynsz 25 000', 'language': 'pl', 'contract_id': 'LEASE-001', 'clause_number': '3.1'})
  [Tool] Zgodnie z klauzulą 3.1 umowy LEASE-001 miesięczny czynsz najmu wynosi **25 000 PLN netto** (plus **VAT 23%**).
  [AI] Aktualny miesięczny czynsz najmu biura wynosi **28 500 PLN netto + 23% VAT**.

**Co zmieniło się w aneksie:**
- Aneks zmienił **pkt 3.1** (czynsz),
- podwyższając czynsz z **25 000 PLN netto + 23% VAT** (umowa bazowa) do **28 500 PLN netto + 23% VAT**,
- a zmiana obowiązuje od **2025-09-01**.
------------------------------------------------------------


### Query 5: Cross-contract comparison

In [8]:
ask('Which contracts have been amended and what were the key changes?');

User: Which contracts have been amended and what were the key changes?
------------------------------------------------------------


  [AI] Tool call: count_contract_documents({})
  [Tool] Total documents=9, contracts=5, amendments=4, unique_contract_ids=5. By language: en(docs=6,contracts=3,amendments=3), pl(docs=3,contracts=2,amendments=1).
  [AI] Tool call: search_contracts({'query': 'amendment key changes', 'document_type': 'amendment'})
  [Tool] ## ITSVC-001 — Amendment No. 2 (effective 2026-01-01)
- **New service tier added (new clause 1.4):** Dedicated **AI/ML engineering service tier** covering model development, MLOps pipeline setup, and ...
  [AI] Tool call: search_contracts({'query': 'Amendment', 'document_type': 'amendment'})
  [Tool] Two amendments are shown:

- **ITSVC-001 — Amendment No. 2 (Version 3), effective 2026-01-01** between **TechFlow Solutions Ltd. (Provider)** and **GlobalRetail Corp. (Client)**. Key changes:
  - Adds...
  [AI] Tool call: search_contracts({'query': 'amends clause', 'document_type': 'amendment'})
  [Tool] - **SLA-001 (Amendment No. 1, effective 2025-10-01):** Amends **clause

## Summary

The agent demonstrates:
- **Amendment awareness**: Returns the latest effective terms, not outdated base contract values
- **Version history**: Can list chronological changes across contract versions
- **Tool-calling with filters**: Can filter by `contract_id`, `language`, `source_type`
- **Multilingual support**: Works with both English and Polish agreements
- **Observability**: All traces are captured in LangFuse (if configured)

For interactive use, run `python scripts/chat.py` from the terminal.